In [ ]:
# Import from this downloaded repository, without pip install -e .
from pathlib import Path
import sys
_start_dir = Path.cwd().resolve()
repo_root = next((p for p in (_start_dir, *_start_dir.parents)
                  if (p / 'src' / 'openvsppy' / '__init__.py').is_file()), None)
if repo_root is None:
    raise FileNotFoundError('Launch this notebook from inside the extracted OpenVSP-main repository.')
source_dir = str(repo_root / 'src')
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)


In [ ]:
import sys
import os

import numpy as np
import pandas as pd
from pathlib import Path
import platform
import psutil
import pprint

# リポジトリの src/openvsppy/vspaero/AnalysisVSPAERO.py をインポート

from openvsppy.vspaero.AnalysisVSPAERO import *

import openvsp as vsp

import os
print(f"Available CPUs: {os.cpu_count()}")


In [ ]:

def get_system_info():
    info = {}
    info['OS'] = platform.system()
    info['OS Version'] = platform.version()
    info['Architecture'] = platform.machine()
    info['Processor'] = platform.processor()
    info['CPU Cores'] = psutil.cpu_count(logical=False)
    info['Logical CPU Cores'] = psutil.cpu_count(logical=True)
    info['Memory (Total)'] = f"{round(psutil.virtual_memory().total / (1024**3), 2)} GB"
    info['Memory (Available)'] = f"{round(psutil.virtual_memory().available / (1024**3), 2)} GB"
    
    return info

system_info = get_system_info()
pprint.pprint(system_info)

In [ ]:

# Clear the current VSP model and read the new VSP file
vsp.ClearVSPModel()
vsp.Update()
vsp.ReadVSPFile(str(repo_root / "examples/notebooks/trimmed_ground_effect/SampleGlider.vsp3"))
vsp.Update()

# Define the list of alpha angles, the Mach number and Reynolds number for the sweep
alpha = [0, 3, -3]
mach = [0.029]
reynolds = [4.3e5]

# Define the list of height
bref = 27
height = list((1-np.cos(np.linspace(0,1,12)*np.pi/2))*bref)[1:]

df = vsp_sweep_wig(
    vsp,
    alpha,
    mach,
    reynolds,
    height,
    analysis_method=0,
    trimmed=False,
    verbose=2,
    # ncpu=12,
    cmy_tolerance=1e-3,
    max_trim_evaluations=6,
    trim_step_deg=2.0,
    wake_num_iter=20,
)

In [ ]:

df = make_CDo_correction(
  vsp, 
  df, 
  Weight=100, 
  xTr=(0.5, 0.7), 
  # CDpCL=0.0016, 
  # thickness=0.19, 
  # interference_factor=1.14, 
  altitude=0, 
  dT=0
)


In [ ]:
df.to_csv('SampleGlider.GroundEffect.polar')
